In [1]:
import json
from typing import Dict, Set

import pandas as pd
import yaml
from IPython.display import display
from rapidfuzz import fuzz

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

psg_directory = "../data/geography/"
psg_data_file = "psgc_2025-07-31.csv"

In [ ]:
df = pd.read_csv(psg_directory + psg_data_file)
display(df.info())
display(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43769 entries, 0 to 43768
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   psgc_id                43769 non-null  int64  
 1   name                   43769 non-null  object 
 2   correspondence_code    43719 non-null  float64
 3   geographic_level       43767 non-null  object 
 4   old_names              1699 non-null   object 
 5   city_class             149 non-null    object 
 6   income_classification  1724 non-null   object 
 7   settlement_type        42011 non-null  object 
 8   population             43762 non-null  object 
 9   Unnamed: 9             76 non-null     object 
 10  barangay_status        2855 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 3.7+ MB


None

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status
0,1300000000,National Capital Region (NCR),130000000.0,Reg,NaN,NaN,NaN,NaN,"13,484,462",NaN,NaN
1,1380100000,City of Caloocan,137501000.0,City,NaN,HUC,1st,NaN,"1,661,584",NaN,NaN
2,1380100001,Barangay 1,137501001.0,Bgy,NaN,NaN,NaN,U,"2,319",NaN,NaN
3,1380100002,Barangay 2,137501002.0,Bgy,NaN,NaN,NaN,U,"5,156",NaN,NaN
4,1380100003,Barangay 3,137501003.0,Bgy,NaN,NaN,NaN,U,"2,497",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
43764,1999908006,Manaulanan,124712037.0,Bgy,NaN,NaN,NaN,U,"7,632",NaN,NaN
43765,1999908007,Pamalian,124712062.0,Bgy,NaN,NaN,NaN,R,"3,256",NaN,NaN
43766,1999908008,Tapodoc,124717017.0,Bgy,NaN,NaN,NaN,R,"1,767",NaN,NaN
43767,1999908009,Macabual,124712034.0,Bgy,NaN,NaN,NaN,R,"4,557",NaN,NaN


In [3]:
# this code is just copied from my barangay project so there are more explanations there
# i think

df["psgc_id"] = df["psgc_id"].astype(str).str.zfill(10)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

geographic_level_map = {
    "Reg": "region",
    "City": "city",
    "Mun": "municipality",
    "Prov": "province",
    "SubMun": "submunicipality",
    "Bgy": "barangay",
}
df["geographic_level"] = df["geographic_level"].replace(geographic_level_map)

df["barangay_code"] = df["psgc_id"].str[-3:]
df["municipal_or_city_code"] = df["psgc_id"].str[-5:-3]
df["province_or_huc_code"] = df["psgc_id"].str[-8:-5]
df["region_code"] = df["psgc_id"].str[-10:-8]

df["barangay_mapper"] = df["psgc_id"].str[-10:]
df["municipal_or_city_mapper"] = df["psgc_id"].str[-10:-3]
df["province_or_huc_mapper"] = df["psgc_id"].str[-10:-5]
df["region_mapper"] = df["psgc_id"].str[-10:-8]

df.sample(10)

regions_filter = (
    (df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)
regions_mapper = (
    df.loc[regions_filter, ["region_mapper", "name"]]
    .sort_values("region_mapper")
    .set_index("region_mapper", drop=True)
    .to_dict()["name"]
)


province_or_huc_filter = (
    ~(df["province_or_huc_code"] == "000")
    & (df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

province_or_huc_mapper = (
    df.loc[province_or_huc_filter, ["province_or_huc_mapper", "name"]]
    .sort_values("province_or_huc_mapper")
    .set_index("province_or_huc_mapper")
    .to_dict()["name"]
)
municipal_or_city_filter = (
    ~(df["province_or_huc_code"] == "000")
    & ~(df["municipal_or_city_code"] == "00")
    & (df["barangay_code"] == "000")
)

municipal_or_city_mapper = (
    df.loc[municipal_or_city_filter, ["municipal_or_city_mapper", "name"]]
    .sort_values("municipal_or_city_mapper")
    .set_index("municipal_or_city_mapper")
    .to_dict()["name"]
)

df["region"] = df["region_mapper"].map(regions_mapper)
df["province_or_huc"] = df["province_or_huc_mapper"].map(province_or_huc_mapper)
df["municipality_or_city"] = df["municipal_or_city_mapper"].map(
    municipal_or_city_mapper
)
display(df.sample(10))

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city
9362,0301414007,Pag-asa,31414007.0,barangay,NaN,NaN,NaN,R,"3,448",NaN,Pob.,007,14,014,03,0301414007,0301414,03014,03,Region III (Central Luzon),Bulacan,Obando
34186,0907221002,Galingon,97221002.0,barangay,NaN,NaN,NaN,R,"1,637",NaN,NaN,002,21,072,09,0907221002,0907221,09072,09,Region IX (Zamboanga Peninsula),Zamboanga del Norte,Tampilisan
37576,1004326005,Kimaya,104326005.0,barangay,NaN,NaN,NaN,R,"1,078",NaN,NaN,005,26,043,10,1004326005,1004326,10043,10,Region X (Northern Mindanao),Misamis Oriental,Villanueva
42528,1903626027,New Lumbacaingud,153626027.0,barangay,NaN,NaN,NaN,R,541,NaN,NaN,027,26,036,19,1903626027,1903626,19036,19,Bangsamoro Autonomous Region In Muslim Mindana...,Lanao del Sur,Tamparan
37567,1004325014,San Jose,104325014.0,barangay,NaN,NaN,NaN,R,"3,763",NaN,NaN,014,25,043,10,1004325014,1004325,10043,10,Region X (Northern Mindanao),Misamis Oriental,Talisayan
16504,1705107002,Harrison,175107002.0,barangay,NaN,NaN,NaN,R,"4,751",NaN,NaN,002,07,051,17,1705107002,1705107,17051,17,MIMAROPA Region,Occidental Mindoro,Paluan
38310,1102507033,Tagugpo,112507033.0,barangay,NaN,NaN,NaN,R,"2,278",NaN,NaN,033,07,025,11,1102507033,1102507,11025,11,Region XI (Davao Region),Davao Oriental,Lupon
24547,0607901005,Bacjao,67901005.0,barangay,Calumingan,NaN,NaN,R,709,NaN,NaN,005,01,079,06,0607901005,0607901,06079,06,Region VI (Western Visayas),Guimaras,Buenavista
6340,0105546018,Dilan Paurido,15546018.0,barangay,NaN,NaN,NaN,U,"7,391",NaN,NaN,018,46,055,01,0105546018,0105546,01055,01,Region I (Ilocos Region),Pangasinan,City of Urdaneta
9182,0301407010,Frances,31407010.0,barangay,NaN,NaN,NaN,U,"6,129",NaN,NaN,010,07,014,03,0301407010,0301407,03014,03,Region III (Central Luzon),Bulacan,Calumpit


In [4]:
clean_mun = (
    df["municipality_or_city"]
    .astype(str)
    .str.lower()
    .str.replace(" ", "")
    .str.replace("(POB.)", "")
)
clean_name = (
    df["name"].astype(str).str.lower().str.replace(" ", "").str.replace("(POB.)", "")
)

df["candidate_hook"] = clean_mun + clean_name

In [5]:
df.sample(10)

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city,candidate_hook
38970,1204703001,Aringay,124703001.0,barangay,NaN,NaN,NaN,R,"4,373",NaN,NaN,001,03,047,12,1204703001,1204703,12047,12,Region XII (SOCCSKSARGEN),Cotabato,Kabacan,kabacanaringay
36901,1004210019,Embargo,104210019.0,barangay,NaN,NaN,NaN,R,817,NaN,NaN,019,10,042,10,1004210019,1004210,10042,10,Region X (Northern Mindanao),Misamis Occidental,City of Ozamiz,cityofozamizembargo
16432,1705101005,San Vicente,175101005.0,barangay,NaN,NaN,NaN,R,"3,162",NaN,NaN,005,01,051,17,1705101005,1705101,17051,17,MIMAROPA Region,Occidental Mindoro,Abra De Ilog,abradeilogsanvicente
3849,0102912022,Maratudo,12912022.0,barangay,NaN,NaN,NaN,R,"1,319",NaN,NaN,022,12,029,01,0102912022,0102912,01029,01,Region I (Ilocos Region),Ilocos Sur,Magsingal,magsingalmaratudo
22145,0600614040,Rizal,60614040.0,barangay,NaN,NaN,NaN,R,340,NaN,NaN,040,14,006,06,0600614040,0600614,06006,06,Region VI (Western Visayas),Antique,San Remigio,sanremigiorizal
29611,0803708095,Villa Mag-aso,83708095.0,barangay,NaN,NaN,NaN,R,613,NaN,NaN,095,08,037,08,0803708095,0803708,08037,08,Region VIII (Eastern Visayas),Leyte,City of Baybay,cityofbaybayvillamag-aso
16608,1705201006,Catwiran II,175201006.0,barangay,NaN,NaN,NaN,R,"1,507",NaN,NaN,006,01,052,17,1705201006,1705201,17052,17,MIMAROPA Region,Oriental Mindoro,Baco,bacocatwiranii
10596,0305404005,Calibutbut,35404005.0,barangay,NaN,NaN,NaN,U,"12,701",NaN,NaN,005,04,054,03,0305404005,0305404,03054,03,Region III (Central Luzon),Pampanga,Bacolor,bacolorcalibutbut
37008,1004213025,Medallo,104213025.0,barangay,NaN,NaN,NaN,R,812,NaN,NaN,025,13,042,10,1004213025,1004213,10042,10,Region X (Northern Mindanao),Misamis Occidental,Sapang Dalaga,sapangdalagamedallo
38600,1108209027,Bayabas,118209027.0,barangay,NaN,NaN,NaN,R,852,NaN,NaN,027,09,082,11,1108209027,1108209,11082,11,Region XI (Davao Region),Davao de Oro,Nabunturan,nabunturanbayabas


In [6]:
from typing import List


def sanitize_input(input_str: str, exclude: List[str] | str | None = None) -> str:
    """
    Removes whitespaces, lowers, and remove all strings listed in exclude
    """
    sanitized_str = input_str.lower()
    if exclude is None:
        return sanitized_str

    if isinstance(exclude, list):
        exclude = [x.lower() for x in exclude if isinstance(x, str)]
        for item in exclude:
            sanitized_str.replace(item, "")
        return sanitized_str

    return sanitized_str.replace(exclude.lower(), "")

In [7]:
input_str = "BACARRALIBTONG"
sanitized_input = sanitize_input(input_str)

df["sanitized_candidate_hook"] = df["candidate_hook"].apply(
    sanitize_input, args=("(pob.)",)
)
df["score"] = (
    df["sanitized_candidate_hook"].apply(fuzz.ratio, args=(sanitized_input,)).round(1)
)

In [8]:
df["score"].value_counts().reset_index().sort_values(by="score", ascending=False)

,score,count
169,100.0,1
183,80.0,1
168,76.9,1
167,74.3,1
154,72.0,2
...,...,...
184,8.7,1
171,8.3,1
165,7.7,2
186,7.4,1


I used excel to parse the PDF... after long battle, didn't work...

Now let's use tabula-py (port of tabula from java)

In [9]:
# let's try tabula-py
import tabula

In [ ]:
df = tabula.read_pdf("../data/education/masterlist.pdf", pages="all")

Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


In [ ]:
from tqdm import tqdm

root_df = pd.DataFrame()
for idx, d in enumerate(tqdm(df)):
    d["page"] = idx
    root_df = pd.concat([root_df, d])


100%|██████████| 544/544 [00:02<00:00, 182.36it/s] 


In [ ]:
root_df = root_df.reset_index(drop=True)

In [ ]:
educdf = root_df[root_df["BEIS School ID"].str.strip().str.isnumeric().notna()]

# Data Cleaning!

In [ ]:
root_df.sample(10)

,Region,Division,District,BEIS School ID,School Name,Street Address,Municipality,Legislative District,Barangay,Sector,Urban/Ru,Sacl hColaosls Sifuicbactliaosnsification,Modified Curricural Offering Classification,page
14278,Region IV-A,Quezon,Catanauan,301311,Doongan Ilaya National High School,N/A,CATANAUAN,3rd District,DOONGAN ILAYA,Public,Partially U,bDaenpED Managed,JHS with SHS,127
5304,Region II,Isabela,Tumauini Sou,h103891,Balug Elementary School,"BALUG, TUMAUINI, ISABELA",TUMAUINI,1st District,BALUG,Public,Partially U,bDaenpED Managed,Purely ES,47
60324,NCR,Las Piñas City,Las Piñas City I,408287,The Little Apprentice Preschool Inc.,"Ground Floor, EVIA North, Daang Hari",CITY OF LAS PIÑAS,Lone District,ALMANZA DOS,Private,Urban,Non-Sectarian,Purely ES,538
11136,Region III,Tarlac City,Tarlac Central,istrict489501,"Holy Triune God Learning School, Inc.",514 Blk 5,CITY OF TARLAC (Capital),2nd District,SAN NICOLAS,Private,Partially U,bNaonn-Sectarian,Purely ES,99
55057,BARMM,Sulu,Talipao,217063,Taraji Primary School,-,TALIPAO,1st District,LOWER KAMUNTAYAN,Public,Partially U,bDaenpED Managed,Purely ES,491
60006,NCR,Pasig City,Pasig City Dist,ict V485613,"Northridge Grade School and Therapy Center, Inc.",59 Kalinangan Street corner C. Raymundo Avenue,CITY OF PASIG,Lone District,CANIOGAN,Private,Urban,Non-Sectarian,Purely ES,535
45259,Region XI,Davao De Oro,Laak,128285,Bayanihan ES,"Purok 1, Bayanihan",LAAK (SAN VICENTE),2nd District,EL KATIPUNAN,Public,Partially U,bDaenpED Managed,Purely ES,404
12195,Region IV-A,Batangas,Mabini,107463,San Teodoro Elementary School,"San Teodoro, Mabini, Batangas",MABINI,2nd District,SAN TEODORO,Public,Partially U,bDaenpED Managed,Purely ES,108
28664,Region VI,Negros Occidental,Moises Padilla,302646,Guinpana-an NHS,PUROK 3,MOISES PADILLA (MAGALLON),5th District,GUINPANA-AN,Public,Partially U,bDaenpED Managed,JHS with SHS,255
60556,NCR,Malabon City,Malabon Distr,ct III487506,Malabon Educational Institution -Arellano Univ...,"Gov. Pascual Avenue, Malabon City",CITY OF MALABON,Lone District,BARITAN,Private,Urban,Non-Sectarian,Purely ES,540


In [ ]:
for col in root_df.columns:
    print(col)

Region
Division
District
BEIS School ID
School Name
Street Address
Municipality
Legislative District
Barangay
Sector
Urban/Ru
Sacl hColaosls Sifuicbactliaosnsification
Modified Curricural Offering Classification
page


In [ ]:
correct_column_names = {
    "Region": "region",
    "Division": "division",
    "District": "district",
    "BEIS School ID": "beis_school_id",
    "School Name": "school_name",
    "Street Address": "street_address",
    "Municipality": "municipality",
    "Legislative District": "legislative_district",
    "Barangay": "barangay",
    "Sector": "sector",
    "Urban/Ru": "settlement_type",
    "Sacl hColaosls Sifuicbactliaosnsification": "school_subclassification",
    "Modified Curricural Offering Classification": "modified_cultural_offering_classification",
    "page": "page",
}

In [ ]:
root_df = root_df.rename(correct_column_names, axis=1)

# Fixing Categories
There are a few categorical data in here that got messed up during the parsing of the PDF File

In [ ]:
root_df.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
13499,Region IV-A,Laguna,Alaminos,402630,Saint Therese of the Child Jesus School (main),"#15 Topaz St., St. Francis Homes 3 San Pedro, ...",SAN PEDRO,1st District,SAN ANTONIO,Private,Urban,Sectarian,All Offering (K to 12),120
23243,Region V,Masbate,Baleno,302144,Magdalena National High School,Magdalena,BALENO,2nd District,MAGDALENA,Public,Partially U,bDaenpED Managed,JHS with SHS,207
56749,CAR,Benguet,Tuba,135685,Andolor ES,Andolor,TUBA,Lone District,TABAAN SUR,Public,Partially U,bDaenpED Managed,Purely ES,506
21357,Region V,Camarines Norte,Jose Panganib,n West112168,San Martin ES,-Barangay San Martin,JOSE PANGANIBAN,1st District,SAN MARTIN,Public,Partially U,bDaenpED Managed,Purely ES,190
23201,Region V,Masbate,Aroroy West,113406,Macabug ES,None,AROROY,2nd District,MACABUG,Public,Partially U,bDaenpED Managed,Purely ES,207
49012,Region XII,Sarangani,South Malung,n130613,Malungon Central Elementary School SPED Center,Poblacion,MALUNGON,Lone District,POBLACION,Public,Partially U,bDaenpED Managed,Purely ES,437
6649,Region III,Bataan,Abucay,104537,P. Rubiano ES,P. Sacdalan,ABUCAY,1st District,MABATANG,Public,Partially U,bDaenpED Managed,Purely ES,59
39756,Region IX,Zamboanga del Norte,Salug I,124623,Salug CS,RAMON MAGSAYSAY,SALUG,3rd District,POBLACION EAST,Public,Partially U,bDaenpED Managed,Purely ES,355
37796,Region VIII,Northern Samar,Pambujan II,123174,Don Sixto Balanquit Elementary School,Purok I,PAMBUJAN,2nd District,"SIXTO T. BALANGUIT, SR.",Public,Rural,DepED Managed,Purely ES,337
16522,Region IV-A,San Pablo City,Sto. Angel,109803,Antonia Manuel Magcase Elementary School,Brgy. Sta. Isabel,SAN PABLO CITY,3rd District,SANTA ISABEL,Public,Urban,DepED Managed,Purely ES,147


In [ ]:
root_df["sector"].value_counts(dropna=False)
# looks clean, bet lets convert to snake case

sector
Public       47421
Private      13256
SUCs/LUCs      247
Name: count, dtype: int64

In [ ]:
root_df["sector"] = root_df["sector"].replace("Public", "public")
root_df["sector"] = root_df["sector"].replace("Private", "private")
root_df["sector"] = root_df["sector"].replace("SUCs/LUCs", "suc_luc")

In [ ]:
root_df.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
46460,Region XI,Davao Oriental,Gov. Generos,South129288,Aguinaldo Elementary School,PUROK 1,GOVERNOR GENEROSO,2nd District,SUROP,public,Partially U,bDaenpED Managed,Purely ES,414
37343,Region VIII,Northern Samar,San Roque,123245,SAN ROQUE CS,SAN ISIDRO ST.,SAN ROQUE,2nd District,ZONE 5 (POB.),public,Partially U,bDaenpED Managed,Purely ES,333
52788,CARAGA,Dinagat Island,Dinagat,324406,Primitivo J. Ebol Memorial National High School,"Magsaysay, Dinagat, Dinagat Islands",DINAGAT,Lone District,MAGSAYSAY,public,Partially U,bDaenpED Managed,JHS with SHS,471
9578,Region III,Tarlac,Capas West,160018,Manabayukan ES,Manabayukan,CAPAS,1st District,O'DONNELL,public,Partially U,bDaenpED Managed,Purely ES,85
24091,Region V,Sorsogon,Castilla West,114121,Canjela ES,"Canjela, Castilla, Sorsogon",CASTILLA,1st District,CANJELA,public,Partially U,bDaenpED Managed,Purely ES,215
40052,Region IX,Zamboanga del Norte,Tampilisan,124862,Tampilisan CS,"Pob. tampilisan, Z.N",TAMPILISAN,3rd District,POBLACION (TAMPILISAN),public,Partially U,bDaenpED Managed,Purely ES,357
59483,NCR,Caloocan City,Caloocan Nort,IV483565,"St. Teresa of Avila Academy, Inc.","Block 7 Lot 1 Phase IV, Tierra Nova, Bagumbong",KALOOKAN CITY,1st District,BARANGAY 171,private,Urban,Non-Sectarian,ES and JHS (K to 10),531
39788,Region IX,Zamboanga del Norte,Sergio Osmeñ,I124648,San Jose ES,"SAN JOSE,SERGIO OSMEÑA SR.",SERGIO OSMEÑA SR.,1st District,SAN JOSE,public,Partially U,bDaenpED Managed,Purely ES,355
51618,CARAGA,Butuan City,East Butuan D,strict II132035,Mahayahay ES,"-Purok-2 Mahayahay, Anticala, Butuan City",BUTUAN CITY (Capital),1st District,ANTICALA,public,Partially U,bDaenpED Managed,Purely ES,460
25894,Region VI,Antique,San Jose,438513,Advance Central College,Salazar Street,SAN JOSE (Capital),Lone District,BARANGAY 1 (POB.),private,Partially U,bNaonn-Sectarian,JHS with SHS,231


In [ ]:
root_df["settlement_type"].value_counts(dropna=False)
# lets correct categories

settlement_type
Partially U    47606
Urban          10404
Rural           2914
Name: count, dtype: int64

In [ ]:
root_df["settlement_type"] = root_df["settlement_type"].replace(
    "Partially U", "partially_urban"
)
root_df["settlement_type"] = root_df["settlement_type"].replace("Urban", "urban")
root_df["settlement_type"] = root_df["settlement_type"].replace("Rural", "rural")


In [ ]:
root_df["settlement_type"].value_counts()

settlement_type
partially_urban    47606
urban              10404
rural               2914
Name: count, dtype: int64

In [ ]:
root_df["school_subclassification"].value_counts(dropna=False)
# now, that's dirty

school_subclassification
bDaenpED Managed                 40457
DepED Managed                     6795
Non-Sectarian                     5244
bNaonn-Sectarian                  4252
bSeacntarian                      2595
Sectarian                         1163
bSUanC Managed                     160
bLoacnal Government                109
SUC Managed                         50
Local Government                    45
bLUanC                              21
LUC                                 16
bDaOnST Managed                      9
DOST Managed                         4
bLoacnal International School        2
bOatnher GA Managed                  1
Other GA Managed                     1
Name: count, dtype: int64

Here, I have to double check the real values in the PDF via reading it manually. Then I'll create the dictionary renamer later

In [ ]:
root_df[root_df["school_subclassification"] == "bSUanC Managed"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
22727,Region V,Camarines Sur,Sipocot North,600056,Central Bicol State University of Agriculture ...,"Coloy-coloy, Impig, Sipocot",SIPOCOT,1st District,IMPIG,suc_luc,partially_urban,bSUanC Managed,JHS with SHS,202
994,Region I,La Union,Agoo East,600004,Don Mariano Marcos Memorial State University-S...,Consolacion,AGOO,2nd District,CONSOLACION (POB.),suc_luc,partially_urban,bSUanC Managed,All Offering (K to 12),8


In [ ]:
root_df[root_df["school_subclassification"] == "bLoacnal Government"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
26491,Region VI,Capiz,Pilar,501100,Guise Integrated School,GUISE,PILAR,1st District,DULANGAN,public,partially_urban,bLoacnal Government,ES and JHS (K to 10),236
26522,Region VI,Capiz,President Rox,s115709,Bayuyan ES,"BAYUYAN, PRES. ROXAS",PRESIDENT ROXAS,1st District,BAYUYAN,public,partially_urban,bLoacnal Government,Purely ES,236


In [ ]:
root_df[root_df["school_subclassification"] == "bDaOnST Managed"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
51566,CARAGA,Butuan City,Southeast I Bu,uan District305490,Philippine Science High School - Caraga Region...,"Tiniwisan, Butuan City",BUTUAN CITY (Capital),1st District,TINIWISAN,public,partially_urban,bDaOnST Managed,JHS with SHS,460
50573,Region XII,Koronadal City,Koronadal We,t District I330521,Philippine Science High School - SOCCSKSARGEN ...,Not Applicable,CITY OF KORONADAL (Capital),2nd District,PARAISO,public,partially_urban,bDaOnST Managed,JHS with SHS,451


In [ ]:
root_df[root_df["school_subclassification"] == "bLoacnal International School"].sample(
    2
)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
21400,Region V,Camarines Norte,Labo East,409756,"ADR Bicol International Technological College,...",P-4,LABO,1st District,MALASUGUI,private,partially_urban,bLoacnal International School,Purely SHS,191
21401,Region V,Camarines Norte,Labo East,409757,"Camarines Norte International School, Inc.","Maharlika Highway, P-1",LABO,1st District,MASALONG,private,partially_urban,bLoacnal International School,Purely SHS,191


In [ ]:
root_df[root_df["school_subclassification"] == "bOatnher GA Managed"].sample()

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
6048,Region II,Tuguegarao City,Tuguegarao,est Educatio1n0a0l 9Z9o4,eDepartment of Agriculture R02 Child Developme...,"Nursery Compound, San Gabriel, Tuguegarao City",TUGUEGARAO CITY(Capital),3rd District,SAN GABRIEL,public,partially_urban,bOatnher GA Managed,Purely ES,54


In [ ]:
mapper = {
    "bDaenpED Managed": "deped_managed",
    "DepED Managed": "deped_managed",
    "Non-Sectarian": "non_sectarian",
    "bNaonn-Sectarian": "non_sectarian",
    "bSeacntarian": "sectarian",
    "Sectarian": "sectarian",
    "bSUanC Managed": "suc_managed",
    "bLoacnal Government": "local_government",
    "SUC Managed": "suc_managed",
    "Local Government": "local_government",
    "bLUanC": "luc",
    "LUC": "luc",
    "bDaOnST Managed": "dost_managed",
    "DOST Managed": "dost_managed",
    "bLoacnal International School": "local_international_school",
    "bOatnher GA Managed": "other_ga_managed",
    "Other GA Managed": "other_ga_managed",
}

# nice list


In [ ]:
# now let's replace

for key, value in mapper.items():
    root_df["school_subclassification"] = root_df["school_subclassification"].replace(
        key, value
    )

In [ ]:
root_df.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
35912,Region VIII,Leyte,Merida,121725,Lundag Elementary School,"Brgy. Lundag, Merida, Leyte",MERIDA,4th District,LUNDAG,public,partially_urban,deped_managed,Purely ES,320
38868,Region VIII,Tacloban City,District Learni,g Center I124242,Tagpuro Elementary School,Tagpuro,TACLOBAN CITY (Capital),1st District,BARANGAY 108 (TAGAPURO),public,urban,deped_managed,Purely ES,347
35945,Region VIII,Leyte,Palo I,404693,Alpha-Omega Learning Center New Life Baptist C...,0507 San Salvador St. St. Michael,PALO,1st District,SAN MIGUEL (POB.),private,partially_urban,non_sectarian,Purely ES,320
38599,Region VIII,Calbayog City,Calbayog Distr,ct III124047,San Jose Elementary School,Purok-2,CALBAYOG CITY,1st District,SAN JOSE,public,partially_urban,deped_managed,Purely ES,344
32857,Region VII,Cebu City,South District,312515,Quiot High School,Sitio Bogo Quiot,CEBU CITY (Capital),2nd District,QUIOT PARDO,public,urban,deped_managed,Purely JHS,293
23618,Region V,Masbate,Placer East,113821,Ban-ao ES,BAN-AO,PLACER,3rd District,BAN-AO,public,partially_urban,deped_managed,Purely ES,210
19759,Region IV-B,Palawan,Taytay II,111206,Alacalian Elementary School,purok 1,TAYTAY,1st District,ALACALIAN,public,partially_urban,deped_managed,Purely ES,176
32262,Region VII,Cebu,Moalboal,119498,Moalboal Central ES,-Pob.West,MOALBOAL,2nd District,POBLACION WEST,public,partially_urban,deped_managed,Purely ES,288
14956,Region IV-A,Quezon,San Francisco,501752,Madagoldol Integrated School,none,SAN FRANCISCO (AURORA),3rd District,INABUAN,public,partially_urban,deped_managed,ES and JHS (K to 10),133
49445,Region XII,South Cotabato,Polomolok W,st468576,"Lapid Kinderland, Inc.",Purok Pag-asa,POLOMOLOK,1st District,MAGSAYSAY,private,partially_urban,non_sectarian,Purely ES,441


In [ ]:
root_df["modified_cultural_offering_classification"].value_counts()

modified_cultural_offering_classification
Purely ES                 43765
JHS with SHS               7490
All Offering (K to 12)     3423
ES and JHS (K to 10)       3056
Purely JHS                 1787
Purely SHS                 1403
Name: count, dtype: int64

In [ ]:
offering_mapper = {
    "Purely ES": "purely_es",
    "JHS with SHS": "jhs_with_shs",
    "All Offering (K to 12)": "all_offering",
    "ES and JHS (K to 10)": "es_and_jhs",
    "Purely JHS": "purely_jhs",
    "Purely SHS": "purely_shs",
}

In [ ]:
root_df["modified_cultural_offering_classification"] = root_df[
    "modified_cultural_offering_classification"
].replace(offering_mapper)

In [ ]:
root_df.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
25939,Region VI,Antique,San Remegio I,115262,Insubuan ES,"-san remigio, antique",SAN REMIGIO,Lone District,INSUBUAN,public,partially_urban,deped_managed,purely_es,231
4873,Region II,Isabela,Palanan,103544,Diddadungan Elementary School,"-Diddadungan,Palanan,Isabela",PALANAN,1st District,DIDDADUNGAN,public,partially_urban,deped_managed,purely_es,43
15976,Region IV-A,Batangas City,Batangas City,ast District 301483,Talumpok Integrated School,Talumpok,BATANGAS CITY (Capital),2nd District,TALUMPOK KANLURAN,public,partially_urban,deped_managed,jhs_with_shs,142
30480,Region VII,Bohol,Catigbian,118181,Rizal PS,Rizal,CATIGBIAN,1st District,RIZAL,public,partially_urban,deped_managed,purely_es,272
36009,Region VIII,Leyte,Palompon So,th121790,Canipaan Elementary School,N/A,PALOMPON,4th District,CANIPAAN,public,partially_urban,deped_managed,purely_es,321
15546,Region IV-A,Rizal,San Mateo,308139,San Mateo National High School - Guinayang Annex,Jurado Comp. Brgy. Guinayang,SAN MATEO,2nd District,GUINAYANG,public,urban,deped_managed,purely_jhs,138
48969,Region XII,Sarangani,East Maitum,130531,Kipalkuda ES,"New La Union, Maitum, Sarangani",MAITUM,Lone District,NEW LA UNION,public,partially_urban,deped_managed,purely_es,437
4098,Region II,Cagayan,Sta. Praxedes,300489,Sta. Praxedes High School,Guerrero St.,SANTA PRAXEDES,2nd District,CENTRO I (POB.),public,partially_urban,deped_managed,jhs_with_shs,36
53,Region I,Ilocos Norte,Badoc,100040,Sta. Cruz ES,"Sta Cruz Norte, Badoc, Ilocos Norte",BADOC,2nd District,SANTA CRUZ SUR,public,partially_urban,deped_managed,purely_es,0
60539,NCR,Malabon City,Malabon Distr,ct II487531,"Academia De La Lilia, Inc.","Block 48 Lot 6 Lapu-Lapu Avenue, corner Hito S...",CITY OF MALABON,Lone District,LONGOS,private,urban,non_sectarian,purely_es,540


In [ ]:
list_of_clean_dfs: List[pd.DataFrame] = []

In [ ]:
root_df["beis_1"] = pd.to_numeric(root_df["beis_school_id"], errors="coerce").astype(
    "Int32"
)

In [ ]:
six_dig_and_not_null = (root_df["beis_1"].astype(str).str.len() == 6) & (
    root_df["beis_1"].notna()
)
root_df[six_dig_and_not_null]

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
0,Region I,Ilocos Norte,Bacarra I,100001,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",BACARRA,1st District,LIBTONG,public,partially_urban,deped_managed,purely_es,0,100001
1,Region I,Ilocos Norte,Bacarra I,100002,Bacarra CES,Santa Rita,BACARRA,1st District,SANTA RITA (POB.),public,partially_urban,deped_managed,purely_es,0,100002
2,Region I,Ilocos Norte,Bacarra I,100003,Buyon ES,NONE,BACARRA,1st District,BUYON,public,partially_urban,deped_managed,purely_es,0,100003
3,Region I,Ilocos Norte,Bacarra I,100004,Ganagan Elementary School,"#37 Ganagan,Bacarra, Ilocos Norte",BACARRA,1st District,GANAGAN,public,partially_urban,deped_managed,purely_es,0,100004
4,Region I,Ilocos Norte,Bacarra I,100005,Macupit ES,Macupit,BACARRA,1st District,MACUPIT,public,partially_urban,deped_managed,purely_es,0,100005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60804,NCR,Taguig,Pateros,488005,SEP Christian School,431 A F. Imson St. San Pedro Pateros Metro Manila,PATEROS,1st District,SAN PEDRO,private,urban,non_sectarian,purely_es,542,488005
60805,NCR,Taguig,Pateros,488043,"Huckleberry Montessori School, Inc. Pateros (M...",A. Almeda St.,PATEROS,1st District,MAGTANGGOL,private,urban,non_sectarian,purely_es,542,488043
60806,NCR,Taguig,Pateros,488112,Maranatha Christian Academy of Tabacalera Pate...,101 F.C. Tuazon Street,PATEROS,1st District,TABACALERA,private,urban,non_sectarian,purely_es,542,488112
60807,NCR,Taguig,Pateros,488114,"ABC Educational Development Center, Inc.",29-A Almeda Street,PATEROS,1st District,MARTIRES DEL 96,private,urban,non_sectarian,purely_es,542,488114


In [ ]:
list_of_clean_dfs.append(root_df[six_dig_and_not_null])

In [ ]:
wdf = root_df[~six_dig_and_not_null]

In [ ]:
wdf.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
38728,Region VIII,Calbayog City,Tinambacan D,strict III501970,Caglanipao Sur Integrated School,Purok-2,CALBAYOG CITY,1st District,CAGLANIPAO SUR,public,partially_urban,deped_managed,es_and_jhs,345,<NA>
44524,Region X,Cagayan de Oro City,Cagayan de O,o City West 4II0 D5i2st3r5i,tMerry Child School,"Zone 7, Bulua, Cagayan de Oro City",CAGAYAN DE ORO CITY (Capital),1st District,BULUA,private,urban,non_sectarian,all_offering,397,<NA>
44521,Region X,Cagayan de Oro City,Cagayan de O,o City West 4II0 D5i2st0r9i,tDiamond Evangelical School Inc.,"Camp Evangelista Patag, Cagayan de Oro City",CAGAYAN DE ORO CITY (Capital),1st District,PATAG,private,urban,sectarian,purely_es,397,<NA>
50215,Region XII,Cotabato City,Cotabato City,istrict III304634,Notre Dame Village National High School,"San Herminigildo St. RH 8, Cotabato City",COTABATO CITY,1st District,ROSARY HEIGHTS VIII,public,urban,deped_managed,jhs_with_shs,448,<NA>
10895,Region III,San Jose del Monte City,San Jose Del,onte West401470,Spirit of Joy School,Main Rd. Cor. San Lorenzo Ruiz St. Pleasant Hi...,CITY OF SAN JOSE DEL MONTE,Lone District,SAN MANUEL,private,urban,non_sectarian,es_and_jhs,97,<NA>
47317,Region XI,Digos City,Digos Occiden,al316303,Balabag National High School,Balabag,CITY OF DIGOS (Capital),1st District,BALABAG,public,partially_urban,deped_managed,purely_jhs,422,<NA>
44667,Region X,Gingoog City,Gingoog City S,uth-2 Distr3ic0t4131,Gingoog City CNHS - BACKKISMI NHS Annex,Purok #1 Binakalan,GINGOOG CITY,1st District,BINAKALAN,public,partially_urban,deped_managed,jhs_with_shs,398,<NA>
36523,Region VIII,Southern Leyte,San Juan (Cab,lian)122249,Dayanog Elementary School,-Municipal road,SAN JUAN (CABALIAN),Lone District,DAYANOG,public,partially_urban,deped_managed,purely_es,326,<NA>
3174,Region I,Urdaneta City,Urdaneta City,istrict II500007,Catablan Integrated School,"Zone 3, Catablan, Urdaneta City, Pangasinan",CITY OF URDANETA,5th District,CATABLAN,public,urban,deped_managed,all_offering,28,<NA>
4479,Region II,Isabela,Angadanan W,st103068,Sinabbaran Elementary School,-PUROK 3,ANGADANAN,3rd District,SINABBARAN,public,partially_urban,deped_managed,purely_es,40,<NA>


In [ ]:
# trial
wseries = wdf.loc[17099]

In [ ]:
import re

re.findall(r"\d+", wseries["beis_school_id"])

['400804']

In [ ]:
# now lets try that
wdf["beis_1"] = wdf["beis_school_id"].str.findall(r"\d+").str[0]

/tmp/ipykernel_8821/2331759299.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wdf["beis_1"] = wdf["beis_school_id"].str.findall(r"\d+").str[0]


In [ ]:
wdf.sample(10)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
59804,NCR,Makati City,Makati City Di,trict V406857,4th Watch Maranatha Christian Academy of Makati,2121 Nuestra Señora St.,CITY OF MAKATI,2nd District,GUADALUPE NUEVO,private,urban,non_sectarian,all_offering,534,406857
54712,BARMM,Maguindanao I,Datu Abdullah,Sangki133874,Campo Cuatro ES,-Campo Cuatro,DATU ABDULLAH SANGKI,2nd District,TALISAWA,public,rural,deped_managed,purely_es,488,133874
16399,Region IV-A,Lucena City,Lucena West,istrict427511,"Growwe Learning Center, Inc.",1170 Garnet Street Iyam Lucena City,LUCENA CITY (Capital),2nd District,ILAYANG IYAM,private,urban,non_sectarian,purely_es,146,427511
40688,Region IX,Zamboanga del Sur,Ramon Magsa,say125322,Magsaysay ES,Purok 2,RAMON MAGSAYSAY (LIARGO),1st District,MAGSAYSAY,public,partially_urban,deped_managed,purely_es,363,125322
17041,Region IV-A,Tanauan City,Tanauan City,orth II321601,Tanauan City Integrated High School,"Trapiche, Tanauan City, Batangas",CITY OF TANAUAN,3rd District,TRAPICHE,public,urban,deped_managed,jhs_with_shs,152,321601
42158,Region X,Malaybalay City,Malaybalay Ci,y District VI 126564,Laguitas ES,"Purok 2, Laguitas, Malaybalay City",CITY MALAYBALAY (Capital),2nd District,LAGUITAS,public,partially_urban,deped_managed,purely_es,376,126564
60524,NCR,Malabon City,Malabon Distr,ct II320501,Longos National High School,"Maya-Maya St., Longos, Malabon City",CITY OF MALABON,Lone District,LONGOS,public,urban,deped_managed,purely_jhs,540,320501
57886,NCR,City of San Juan,San Juan Distr,ct I485503,Fountain International School,"14 Annapolis St., Greenhills",CITY OF SAN JUAN,Lone District,GREENHILLS,private,urban,non_sectarian,purely_es,516,485503
58341,NCR,Quezon City,School District,I406392,Sacred Heart Academy of La Loma,"49 N.S. Amoranto Sr. Ave., La Loma Quezon, City",QUEZON CITY,1st District,PAANG BUNDOK,private,urban,non_sectarian,all_offering,520,406392
18239,Region IV-B,Occidental Mindoro,Abra De Ilog-P,luan305854,Pambuhan Indigenous People Village High School,None,ABRA DE ILOG,Lone District,SAN VICENTE,public,partially_urban,deped_managed,purely_jhs,162,305854


In [ ]:
is_six_digit = (
    pd.to_numeric(wdf["beis_1"], errors="coerce").astype("Int32").astype(str).str.len()
    == 6
)

In [ ]:
wdf[is_six_digit]

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
87,Region I,Ilocos Norte,Banna (Espirit,)100058,Bangsar ES,Bangsar,BANNA (ESPIRITU),2nd District,BANGSAR,public,partially_urban,deped_managed,purely_es,0,100058
88,Region I,Ilocos Norte,Banna (Espirit,)100059,Banna Central Elementary School,"P. Gomez, Marcos, Banna, Ilocos Norte",BANNA (ESPIRITU),2nd District,MARCOS (POB.),public,partially_urban,deped_managed,purely_es,0,100059
89,Region I,Ilocos Norte,Banna (Espirit,)100060,Barbarangay ES,Barbarangay,BANNA (ESPIRITU),2nd District,BARBARANGAY,public,partially_urban,deped_managed,purely_es,0,100060
90,Region I,Ilocos Norte,Banna (Espirit,)100061,Bomitog ES,Banna - Pinili Rd.,BANNA (ESPIRITU),2nd District,BOMITOG,public,partially_urban,deped_managed,purely_es,0,100061
91,Region I,Ilocos Norte,Banna (Espirit,)100062,Bugasi ES,Bugasi,BANNA (ESPIRITU),2nd District,BUGASI,public,partially_urban,deped_managed,purely_es,0,100062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60919,NCR,Muntinlupa City,Muntinlupa Ci,y District II488533,"CBC Integrated School, Inc.","1560 Estanislao Street, Lakeview Homes I",CITY OF MUNTINLUPA,Lone District,PUTATAN,private,urban,non_sectarian,purely_es,543,488533
60920,NCR,Muntinlupa City,Muntinlupa Ci,y District II488544,"The Linden Tree Institute, Inc.",177 Buencamino Street,CITY OF MUNTINLUPA,Lone District,ALABANG,private,urban,non_sectarian,es_and_jhs,543,488544
60921,NCR,Muntinlupa City,Muntinlupa Ci,y District II488547,Cambridge Children's Learning and Development ...,Lower Ground Level Alabang Town Center,CITY OF MUNTINLUPA,Lone District,ALABANG,private,urban,sectarian,purely_es,543,488547
60922,NCR,Muntinlupa City,Muntinlupa Ci,y District II488548,Holy Word Christian School,"4 Cattleya St., Doña Rosario Bayview Subdivision",CITY OF MUNTINLUPA,Lone District,SUCAT,private,urban,sectarian,es_and_jhs,543,488548


In [ ]:
wwdf = wdf[~is_six_digit]

In [ ]:
# wwdf["beis_1"] =
wwdf["beis_1"] = (
    pd.to_numeric(
        wwdf["beis_school_id"].str.findall(r"\d+").str.join(""), errors="coerce"
    )
    .astype("Int32")
    .astype(str)
)

/tmp/ipykernel_8821/645232286.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wwdf["beis_1"] = pd.to_numeric(wwdf["beis_school_id"].str.findall(r"\d+").str.join(""), errors="coerce").astype("Int32").astype(str)


In [ ]:
nuther_six_dig = wwdf["beis_1"].str.len() == 6

In [ ]:
list_of_clean_dfs.append(wwdf[nuther_six_dig])

In [ ]:
# Now's the hard part

In [ ]:
rdf = wwdf[~nuther_six_dig]

In [ ]:
sev_dig = rdf["beis_1"].str.len() == 7

In [ ]:
sev_df = rdf[sev_dig]

In [ ]:
sev_df["beis_1"] = sev_df["beis_1"].str[1:]

/tmp/ipykernel_8821/3908203776.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sev_df["beis_1"] = sev_df["beis_1"].str[1:]


In [ ]:
list_of_clean_dfs.append(sev_df)

In [ ]:
wherf = rdf[~sev_dig]

In [ ]:
wherf["beis_1"] = wherf["beis_school_id"]

/tmp/ipykernel_8821/170272940.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wherf["beis_1"] = wherf["beis_school_id"]


In [ ]:
wherf["beis_1"] = wherf["beis_1"].astype(str).str[1:]

/tmp/ipykernel_8821/2151556274.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wherf["beis_1"] = wherf["beis_1"].astype(str).str[1:]


In [ ]:
list_of_clean_dfs.append(wherf)

In [ ]:
raise KeyboardInterrupt

KeyboardInterrupt: 

In [ ]:
mother_df: pd.DataFrame = pd.DataFrame()
for a_df in list_of_clean_dfs:
    mother_df = pd.concat([mother_df, a_df])

In [ ]:
mother_df["beis_school_id"] = mother_df["beis_1"]

In [ ]:
mother_df["beis_school_id"].astype(str).str.len().value_counts()

beis_school_id
6    60924
Name: count, dtype: int64

In [ ]:
mother_df["page"] = mother_df["page"] + 1

In [ ]:
mother_df = mother_df.rename({"page": "masterlist_page"}, axis=1)

In [ ]:
mother_df = mother_df.drop("beis_1", axis=1)

In [ ]:
mother_df

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page
0,Region I,Ilocos Norte,Bacarra I,100001,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",BACARRA,1st District,LIBTONG,public,partially_urban,deped_managed,purely_es,1
1,Region I,Ilocos Norte,Bacarra I,100002,Bacarra CES,Santa Rita,BACARRA,1st District,SANTA RITA (POB.),public,partially_urban,deped_managed,purely_es,1
2,Region I,Ilocos Norte,Bacarra I,100003,Buyon ES,NONE,BACARRA,1st District,BUYON,public,partially_urban,deped_managed,purely_es,1
3,Region I,Ilocos Norte,Bacarra I,100004,Ganagan Elementary School,"#37 Ganagan,Bacarra, Ilocos Norte",BACARRA,1st District,GANAGAN,public,partially_urban,deped_managed,purely_es,1
4,Region I,Ilocos Norte,Bacarra I,100005,Macupit ES,Macupit,BACARRA,1st District,MACUPIT,public,partially_urban,deped_managed,purely_es,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32715,Region VII,Cebu,Pinamungajan,119572,Tajao Central School,South,PINAMUNGAHAN,3rd District,TAJAO,public,partially_urban,deped_managed,purely_es,293
32716,Region VII,Cebu,Pinamungajan,119574,Tanibag ES,"Tanibag, Pinamungajan, Cebu",PINAMUNGAHAN,3rd District,TANIBAG,public,partially_urban,deped_managed,purely_es,293
32717,Region VII,Cebu,Pinamungajan,187011,Buhingtubig ES,"Buhingtubig, Pinamungajan",PINAMUNGAHAN,3rd District,BUHINGTUBIG,public,partially_urban,deped_managed,purely_es,293
32718,Region VII,Cebu,Pinamungajan,187012,Cabiangon ES,"Cabiangon, Pinamungajan",PINAMUNGAHAN,3rd District,CABIANGON,public,partially_urban,deped_managed,purely_es,293


In [ ]:
mother_df["beis_school_id"] = mother_df["beis_school_id"].astype(str)

In [ ]:
mother_df.to_parquet(
    "../data/education/basic_education_institutions.parquet", index=False
)

In [ ]:
mother_df.to_csv("../data/education/basic_education_institutions.csv", index=False)

# Matching PSGC_ID

In [ ]:
import pandas as pd
from typing import List
import rapidfuzz

from functools import partial


edf = pd.read_parquet("../data/education/basic_education_institutions.parquet")


def sanitize_input(
    input_str: str | None, exclude: List[str] | str | None = None
) -> str:
    """
    Removes whitespaces, lowers, and remove all strings listed in exclude. If
    data is incompatible, will coerce to empty string.
    """
    if input_str is None:
        input_str = ""
    if not isinstance(input_str, str):
        input_str = ""
    sanitized_str = input_str.lower()
    if exclude is None:
        return sanitized_str

    if isinstance(exclude, list):
        exclude = [x.lower() for x in exclude if isinstance(x, str)]
        for item in exclude:
            sanitized_str = sanitized_str.replace(item, "")
        return sanitized_str

    return sanitized_str.replace(exclude.lower(), "")


cleanerjim = partial(
    sanitize_input, exclude=["(pob.)", "(pob)", ".", " ", "-", "(", ")", "&", "pob."]
)

In [ ]:
edf["low_mun"] = edf["municipality"].str.lower()
edf["low_bar"] = edf["barangay"].str.lower()
edf["low_dis"] = edf["district"].str.lower()
edf["low_div"] = edf["division"].str.lower()
edf["low_reg"] = edf["region"].str.lower()

key_candidates = [
    "low_mun",
    "low_dis",
    "low_div",
    "low_reg",
    "low_bar",
]

In [ ]:
for key in key_candidates:
    edf["c_" + key] = edf[key].apply(cleanerjim)

edf.sample(2)

KeyError: 'low_mun'

In [ ]:
edf["00mb"] = edf["c_low_mun"] + edf["c_low_bar"]

In [ ]:
edf.sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page,low_mun,low_bar,low_dis,low_div,low_reg,c_low_mun,c_low_dis,c_low_div,c_low_reg,c_low_bar,00mb
47826,Region III,Balanga City,Balanga City E,400593,Asia Pacific College of Advanced Studies,A. H. Banzon Street,CITY OF BALANGA (Capital),2nd District,IBAYO,private,partially_urban,non_sectarian,all_offering,102,city of balanga (capital),ibayo,balanga city e,balanga city,region iii,cityofbalangacapital,balangacitye,balangacity,regioniii,ibayo,cityofbalangacapitalibayo
14930,Region IV-B,Oriental Mindoro,Roxas,110638,Paclasan ES,Sto.Niño St.,ROXAS,2nd District,PACLASAN (POB.),public,partially_urban,deped_managed,purely_es,171,roxas,paclasan (pob.),roxas,oriental mindoro,region iv-b,roxas,roxas,orientalmindoro,regionivb,paclasan,roxaspaclasan


In [ ]:
old_df = df.copy()

In [ ]:
bdf = old_df[old_df["geographic_level"] == "barangay"]

In [ ]:
bdf["00mb"] = (bdf["municipality_or_city"].astype(str) + bdf["name"].astype(str)).apply(
    cleanerjim
)

/tmp/ipykernel_54182/267393626.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bdf["00mb"] = (bdf["municipality_or_city"].astype(str)+bdf["name"].astype(str)).apply(cleanerjim)


In [ ]:
bdf.sample(2)

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city,00mb,score,ratio_score
30682,0803746009,Mercaduhay,83746009.0,barangay,NaN,NaN,NaN,R,884,NaN,NaN,009,46,037,08,0803746009,0803746,08037,08,Region VIII (Eastern Visayas),Leyte,Tabontabon,tabontabonmercaduhay,29.411765,29.411765
13217,0402102011,Barangay I,42102011.0,barangay,NaN,NaN,NaN,R,"1,551",NaN,Pob.,011,02,021,04,0402102011,0402102,04021,04,Region IV-A (CALABARZON),Cavite,Amadeo,amadeobarangayi,34.482759,34.482759


In [ ]:
# testing shit out

In [ ]:
first_trial = edf.loc[17992]

In [ ]:
first_trial["00mb"]

'aroroygumahang'

In [ ]:
first_trial["00mb"].iloc[0]

'aroroygumahang'

In [ ]:
from functools import partial

fuzzer = partial(rapidfuzz.fuzz.ratio, s2=first_trial["00mb"])


In [ ]:
bdf["fuzzer"] = bdf["00mb"].apply(lambda ref: partial(rapidfuzz.fuzz.ratio, s1=ref))

/tmp/ipykernel_54182/3282512117.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bdf["fuzzer"] = bdf["00mb"].apply(lambda ref: partial(rapidfuzz.fuzz.ratio, s1=ref))


In [ ]:
bdf["ratio_score"]

26565    functools.partial(<cyfunction ratio at 0x7f35f...
16208    functools.partial(<cyfunction ratio at 0x7f35f...
20856    functools.partial(<cyfunction ratio at 0x7f35f...
43515    functools.partial(<cyfunction ratio at 0x7f35f...
25807    functools.partial(<cyfunction ratio at 0x7f35f...
11774    functools.partial(<cyfunction ratio at 0x7f35f...
22915    functools.partial(<cyfunction ratio at 0x7f35f...
5237     functools.partial(<cyfunction ratio at 0x7f35f...
36653    functools.partial(<cyfunction ratio at 0x7f35f...
23752    functools.partial(<cyfunction ratio at 0x7f35f...
Name: fuzzer, dtype: object

In [ ]:
bdf["ratio_score"] = bdf["00mb"].apply(fuzzer)
bdf.sort_values(by="ratio_score", ascending=False).iloc[0:5]

/tmp/ipykernel_54182/1910812255.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bdf["ratio_score"] = bdf["00mb"].apply(fuzzer)


,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city,00mb,score,ratio_score
20218,0504101015,Gumahang,54101015.0,barangay,NaN,NaN,NaN,R,"2,055",NaN,NaN,015,01,041,05,0504101015,0504101,05041,05,Region V (Bicol Region),Masbate,Aroroy,aroroygumahang,57.142857,100.000000
20220,0504101017,Lanang,54101017.0,barangay,NaN,NaN,NaN,R,"1,009",NaN,NaN,017,01,041,05,0504101017,0504101,05041,05,Region V (Bicol Region),Masbate,Aroroy,aroroylanang,46.153846,76.923077
20238,0504101035,Sawang,54101035.0,barangay,NaN,NaN,NaN,R,"1,371",NaN,NaN,035,01,041,05,0504101035,0504101,05041,05,Region V (Bicol Region),Masbate,Aroroy,aroroysawang,46.153846,76.923077
20207,0504101004,Bagauma,54101004.0,barangay,NaN,NaN,NaN,R,"2,841",NaN,NaN,004,01,041,05,0504101004,0504101,05041,05,Region V (Bicol Region),Masbate,Aroroy,aroroybagauma,44.444444,74.074074
20222,0504101019,Macabug,54101019.0,barangay,NaN,NaN,NaN,R,"1,084",NaN,NaN,019,01,041,05,0504101019,0504101,05041,05,Region V (Bicol Region),Masbate,Aroroy,aroroymacabug,44.444444,74.074074


In [ ]:
# creating combinator

In [ ]:
bdf.sample(2)

,psgc_id,name,correspondence_code,geographic_level,old_names,city_class,income_classification,settlement_type,population,Unnamed: 9,barangay_status,barangay_code,municipal_or_city_code,province_or_huc_code,region_code,barangay_mapper,municipal_or_city_mapper,province_or_huc_mapper,region_mapper,region,province_or_huc,municipality_or_city,00mb,score,ratio_score,fuzzer
24554,0607901015,Magsaysay,67901015.0,barangay,NaN,NaN,NaN,R,616,NaN,NaN,015,01,079,06,0607901015,0607901,06079,06,Region VI (Western Visayas),Guimaras,Buenavista,buenavistamagsaysay,30.303030,24.242424,functools.partial(<cyfunction ratio at 0x7f35f...
27430,0702203004,Lepanto,72203004.0,barangay,NaN,NaN,NaN,R,"2,405",NaN,NaN,004,03,022,07,0702203004,0702203,07022,07,Region VII (Central Visayas),Cebu,Alegria,alegrialepanto,28.571429,35.714286,functools.partial(<cyfunction ratio at 0x7f35f...


In [ ]:
fuzzer_base = bdf[["name", "province_or_huc", "municipality_or_city", "psgc_id"]]

In [ ]:
fuzzer_base = fuzzer_base.rename({"name": "barangay"}, axis=1)

In [ ]:
fuzzer_base.to_parquet("../data/geography/fuzzer_base.parquet")

In [ ]:
fuzzer_base["0p0b"] = (fuzzer_base["province_or_huc"] + fuzzer_base["barangay"]).apply(
    cleanerjim
)
fuzzer_base["00mb"] = (
    fuzzer_base["municipality_or_city"].astype(str)
    + fuzzer_base["barangay"].astype(str)
).apply(cleanerjim)
fuzzer_base["0pmb"] = (
    fuzzer_base["province_or_huc"].astype(str)
    + fuzzer_base["municipality_or_city"].astype(str)
    + fuzzer_base["barangay"].astype(str)
).apply(cleanerjim)

fuzzer_base["f_00mb_ratio"] = fuzzer_base["00mb"].apply(
    lambda ref: partial(rapidfuzz.fuzz.ratio, s1=ref)
)
fuzzer_base["f_0p0b_ratio"] = fuzzer_base["0p0b"].apply(
    lambda ref: partial(rapidfuzz.fuzz.ratio, s1=ref)
)
fuzzer_base["f_0pmb_ratio"] = fuzzer_base["0pmb"].apply(
    lambda ref: partial(rapidfuzz.fuzz.ratio, s1=ref)
)


In [ ]:
sample = "Carsadang Bago II"

In [ ]:
match_hooks = ["province", "municipality", "barangay"]
threshold: float = 70.0
number_of_results: int = 1
active_ratios: List[str] = []

cleaned_sample: str = cleanerjim(sample)

# PB
if "province" in match_hooks and "barangay" in match_hooks:
    fuzzer_base["f_0p0b_ratio" + "_score"] = fuzzer_base["f_0p0b_ratio"].apply(
        lambda f: f(s2=cleaned_sample)
    )
    match_on_0p0b = fuzzer_base["f_0p0b_ratio_score"].gt(threshold)
    active_ratios.append("f_0p0b_ratio_score")

# MB
if "municipality" in match_hooks and "barangay" in match_hooks:
    fuzzer_base["f_00mb_ratio" + "_score"] = fuzzer_base["f_00mb_ratio"].apply(
        lambda f: f(s2=cleaned_sample)
    )
    match_on_00mb = fuzzer_base["f_00mb_ratio_score"].gt(threshold)
    active_ratios.append("f_00mb_ratio_score")

# PMB
if (
    "province" in match_hooks
    and "municipality" in match_hooks
    and "barangay" in match_hooks
):
    fuzzer_base["f_0pmb_ratio" + "_score"] = fuzzer_base["f_0pmb_ratio"].apply(
        lambda f: f(s2=cleaned_sample)
    )
    match_on_0pmb = fuzzer_base["f_0pmb_ratio_score"].gt(threshold)
    active_ratios.append("f_0pmb_ratio_score")

fuzzer_base["max_score"] = fuzzer_base[active_ratios].max(axis=1)
result = fuzzer_base.sort_values(by="max_score", ascending=False).iloc[
    0:number_of_results
]
a_variable = result[
    ["barangay", "province_or_huc", "municipality_or_city", "psgc_id"]
].to_dict(orient="records")

In [ ]:
a_variable

[{'barangay': 'Carsadang Bago II',
  'province_or_huc': 'Cavite',
  'municipality_or_city': 'City of Imus',
  'psgc_id': '0402109054'}]

In [ ]:
for x in list(bdf["region"].value_counts().index):
    print(x)

Region VIII (Eastern Visayas)
Region IV-A (CALABARZON)
Region V (Bicol Region)
Region VI (Western Visayas)
Region I (Ilocos Region)
Region III (Central Luzon)
Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)
Region VII (Central Visayas)
Region II (Cagayan Valley)
Region X (Northern Mindanao)
Region IX (Zamboanga Peninsula)
National Capital Region (NCR)
MIMAROPA Region
Negros Island Region (NIR)
Region XIII (Caraga)
Cordillera Administrative Region (CAR)
Region XI (Davao Region)
Region XII (SOCCSKSARGEN)


# Adding PSGC ID to basic_ed Dataset

Because [this](https://www.data.pssc.org.ph/docs/department-of-education/) is not working

In [ ]:
import pandas as pd
from barangay import search, sanitize_input, _basic_sanitizer

bed_df = pd.read_parquet("../data/education/basic_education_institutions.parquet")

In [ ]:
from functools import partial

educ_sanitizer = partial(
    sanitize_input,
    exclude=[
        "(pob.)",
        "(pob)",
        "(capital)",
        "city of",
        "city",
        ".",
        "-",
        "(",
        ")",
        "&",
        "pob.",
        ",",
    ],
)

In [ ]:
bed_df["search_term"] = bed_df["municipality"] + " " + bed_df["barangay"]
bed_df["search_term"] = bed_df["search_term"].apply(educ_sanitizer)

In [ ]:
bed_df.sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page,search_term
17753,Region V,Camarines Sur,Tigaon,433590,"Hansel & Gretel Foundation, Inc.","Zone 7, Sitio Bulala",TIGAON,4th District,COYAOYAO,private,partially_urban,non_sectarian,purely_es,204,tigaon coyaoyao
10571,Region IV-A,Cavite,Tagaytay City,108150,Patutong Malaki Elementary School,Brgy. Patutong Malaki South,TAGAYTAY CITY,7th District,PATUTONG MALAKI SOUTH,public,urban,deped_managed,purely_es,119,tagaytay patutong malaki south


In [ ]:
import swifter

from functools import partial

custom_search = partial(
    search,
    match_hooks=["municipality", "barangay"],
    threshold=0.97,
    n=1,
    sanitizer=educ_sanitizer,
)

bed_df["score"] = (
    bed_df["search_term"].swifter.progress_bar(True, "progress").apply(custom_search)
)

progress:   0%|          | 0/60924 [00:00<?, ?it/s]

In [ ]:
bed_df.to_parquet(".parquet")

In [ ]:
bed_df.sample()["score"].iloc[0]

[{'barangay': 'San Jacinto',
  'province_or_huc': 'Tarlac',
  'municipality_or_city': 'San Manuel',
  'psgc_id': '0306914011'}]

In [ ]:
bed_df["result_length"] = bed_df["score"].apply(lambda x: len(x))

In [ ]:
bed_df["psgc_province_or_huc"] = bed_df["score"].apply(
    lambda x: x[0]["province_or_huc"] if len(x) > 0 else ""
)
bed_df["psgc_municipality_or_city"] = bed_df["score"].apply(
    lambda x: x[0]["municipality_or_city"] if len(x) > 0 else ""
)
bed_df["psgc_barangay"] = bed_df["score"].apply(
    lambda x: x[0]["barangay"] if len(x) > 0 else ""
)
bed_df["psgc_id"] = bed_df["score"].apply(
    lambda x: x[0]["psgc_id"] if len(x) > 0 else ""
)

In [ ]:
bed_df[
    [
        "barangay",
        "municipality",
        "psgc_municipality_or_city",
        "psgc_barangay",
        "psgc_id",
    ]
].sample(20)

,barangay,municipality,psgc_municipality_or_city,psgc_barangay,psgc_id
57072,CULIAT,QUEZON CITY,City of Lipa,Quezon,0401014052
28011,RIZAL,KANANGA,Kananga,Rizal,0803726016
4403,AMMUGAUAN,SANTO TOMAS,Santo Tomas,Ammugauan,0203136001
39504,KINABJANGAN,NASIPIT,Nasipit,Kinabjangan,1600209009
45151,TRANCOVILLE,BAGUIO CITY,City of Bago,Pacol,1804502019
5068,CENTRO POBLACION,ILAGAN CITY (CAPITAL),City of Ilagan,Centro Poblacion,0203114100
53531,KALASUNGAY,CITY MALAYBALAY (Capital),City of Malaybalay,Kalasungay,1001312019
5220,LAWANG,DILASAG,Dilasag,Lawang,0307703006
5330,QUIRINO,MARIA AURORA,Maria Aurora,Quirino,0307707030
24403,MAHAGBU,TRINIDAD,Trinidad,Mahagbu,0701244013


In [ ]:
bed_df["result_length"].value_counts()

result_length
1    60873
0       51
Name: count, dtype: int64

In [ ]:
import pandas as pd

# Round 2, Fight!

In [17]:
import pandas as pd
from barangay import sanitize_input, _basic_sanitizer
from functools import partial

In [3]:
df = pd.read_parquet("../data/education/basic_ed_matches_round02.parquet")

In [ ]:
df["psgc_province_or_huc"] = df["score"].apply(
    lambda x: x[0]["province_or_huc"] if len(x) > 0 else ""
)
df["psgc_municipality_or_city"] = df["score"].apply(
    lambda x: x[0]["municipality_or_city"] if len(x) > 0 else ""
)
df["psgc_barangay"] = df["score"].apply(
    lambda x: x[0]["barangay"] if len(x) > 0 else ""
)
df["psgc_id"] = df["score"].apply(lambda x: x[0]["psgc_id"] if len(x) > 0 else "")

In [46]:
slim_view = [
    "barangay",
    "municipality",
    "psgc_province_or_huc",
    "psgc_barangay",
    "psgc_municipality_or_city",
    "psgc_id",
]
df[slim_view].sample(10)

,barangay,municipality,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_id
5721,CONCEPCION,BALIUAG,Tarlac,Balutu,Concepcion,0306905003
876,JUAN CARTAS,CABA,La Union,Juan Cartas,Caba,0103309003
11976,MAGSAYSAY,SAN ANTONIO,Quezon,Magsaysay,San Antonio,0405641011
15153,BULALACAO,CORON,Palawan,Bulalacao,Coron,1705309005
22593,SAN PABLO,MANAPLA,Negros Occidental,San Pablo,Manapla,1804518010
11620,VILLA AURORA,LOPEZ,Quezon,Villa Aurora,Lopez,0405622089
37084,CENTRO (SAN JUAN),DAVAO CITY,Ilocos Sur,Darao,San Juan,0102920010
200,SAN ANTONIO,PIDDIG,Ilocos Norte,San Antonio,Piddig,0102818019
60370,LAHING-LAHING,OMAR,Sulu,Lahing-Lahing,Omar,1906619005
41084,NUEVA ESTRELLA,CAGDIANAO,Dinagat Islands,Nueva Estrella,Cagdianao,1608502007


In [6]:
sampled_locations = [
    18927,
    60534,
    36551,
    8737,
    28839,
    5636,
    7777,
    49961,
    21436,
    50025,
]

In [7]:
df.loc[sampled_locations, slim_view]

,barangay,municipality,psgc_barangay,psgc_municipality_or_city,psgc_id
18927,TABOC,JUBAN,Taboc,Juban,0506210025
60534,BARANGAY 9,KALOOKAN CITY,Barangay 9,Lucban,0405623012
36551,BUHANGIN (POB.),DAVAO CITY,Buhangin,Naujan,1705208012
8737,SAPALIBUTAD,ANGELES CITY,Sapalibutad,None,0330100030
28839,BOGO (POB.),TOMAS OPPUS,Bogo,Tomas Oppus,0806418003
5636,BOROL 1ST,BALAGTAS (BIGAA),Borol 1st,Balagtas,0301402003
7777,SAMPUT,PANIQUI,Samput,Paniqui,0306910028
49961,BABUYAN,PUERTO PRINCESA CITY (Capital),Puerto Princesa,Basilisa,1608501014
21436,DATAGAN,CALINOG,Datagan,Calinog,0603013028
50025,SAN FRANCISCO,GUINOBATAN,San Francisco,Guinobatan,0500504043


Hmm, here we can see that
1. Matched
2. Did not match because kalookan city (??? wtf btw, why is it spelled like that)
   1. How to handle, hmm, idk transform ed from kalookan -> caloocan #R02-T001
3. I honestly don't know why
   1. I guess city as well? because davao city vs city of davao?
4. Fucked by ICCs
   1. How to handle, exclude "City" #R02-T002
5. Matched
6. Matched
7. Matched
8. Municipality matched on barangay
   1. How to handle, #R02-T002
9.  Matched
10. Matched

accuracy rate: 6/10



# R02-T001

In [18]:
r02_t001_sanitizer = partial(
    sanitize_input,
    exclude=[
        "(pob.)",
        "(pob)",
        "pob.",
        "(capital)",
        "city of ",
        " city",
        ".",
        "-",
        "(",
        ")",
        "&",
        ",",
    ],
)

In [19]:
r02_t001 = df[df["municipality"].str.contains("KALOOKAN")]
r02_t001.sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page,search_term,score,psgc_province_or_huc,psgc_municipality_or_city,psgc_barangay,psgc_id
60524,NCR,Caloocan City,Pobcaran (Po,136619,Cayetano Arellano Elementary School,"Payapa Cor. Tahimik 6th Ave, Caloocan City",KALOOKAN CITY,2nd District,BARANGAY 125,public,urban,deped_managed,purely_es,529,kalookan barangay 125,"[{'barangay': 'Barangay 125', 'municipality_or...",City of Caloocan,None,Barangay 125,1380100125
57840,NCR,Caloocan City,Caloocan Nort,136649,Tala ES,"Administration Site, Tala",KALOOKAN CITY,1st District,BARANGAY 186,public,urban,deped_managed,purely_es,531,kalookan barangay 186,"[{'barangay': 'Barangay 186', 'municipality_or...",City of Caloocan,None,Barangay 186,1380100186


In [23]:
r02_t001["municipality"] = r02_t001["municipality"].str.replace("KALOOKAN","CALOOCAN")

/tmp/ipykernel_2284/305399862.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["municipality"] = r02_t001["municipality"].str.replace("KALOOKAN","CALOOCAN")


In [24]:
r02_t001["search_term"] = r02_t001["municipality"] + " " + r02_t001["barangay"]
r02_t001["search_term"] = r02_t001["search_term"].apply(r02_t001_sanitizer)

/tmp/ipykernel_2284/1836937612.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["search_term"] = r02_t001["municipality"] + " " + r02_t001["barangay"]
/tmp/ipykernel_2284/1836937612.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["search_term"] = r02_t001["search_term"].apply(r02_t001_sanitizer)


In [25]:
r02_t001

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page,search_term,score,psgc_province_or_huc,psgc_municipality_or_city,psgc_barangay,psgc_id
45492,NCR,Caloocan City,Tanque,136625,Baesa ES,"229 Reparo St., Brgy. 161",CALOOCAN CITY,1st District,BARANGAY 161,public,urban,deped_managed,purely_es,530,caloocan barangay 161,"[{'barangay': 'Barangay 161', 'municipality_or...",City of Caloocan,None,Barangay 161,1380100161
45493,NCR,Caloocan City,Tanque,136626,Libis Baesa Elementary School (Baesa Annex),Blk.8 Lot 40 Libis Baesa,CALOOCAN CITY,1st District,BARANGAY 160,public,urban,deped_managed,purely_es,530,caloocan barangay 160,"[{'barangay': 'Barangay 160', 'municipality_or...",City of Caloocan,None,Barangay 160,1380100160
45494,NCR,Caloocan City,Tanque,136627,Bagong Barrio ES,G. de Jesus St. corner Malolos,CALOOCAN CITY,1st District,BARANGAY 147,public,urban,deped_managed,purely_es,530,caloocan barangay 147,"[{'barangay': 'Barangay 147', 'municipality_or...",Pasay City,None,Barangay 147,1381100147
45495,NCR,Caloocan City,Tanque,136628,East Bagong Barrio ES,79 Tieremas St.,CALOOCAN CITY,1st District,BARANGAY 157,public,urban,deped_managed,purely_es,530,caloocan barangay 157,"[{'barangay': 'Barangay 157', 'municipality_or...",Pasay City,None,Barangay 157,1381100157
45496,NCR,Caloocan City,Tanque,136629,Morning Breeze Elementary School,Pilar St.,CALOOCAN CITY,1st District,BARANGAY 83,public,urban,deped_managed,purely_es,530,caloocan barangay 83,"[{'barangay': 'Barangay 3', 'municipality_or_c...",Batangas,Lian,Barangay 3,0401013018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60548,NCR,Caloocan City,Pobcaran (Po,483609,"Little Einstein Discovery School of Caloocan, ...",101 General Luna Street,CALOOCAN CITY,2nd District,BARANGAY 18,private,urban,non_sectarian,purely_es,530,caloocan barangay 18,"[{'barangay': 'Barangay 1', 'municipality_or_c...",Batangas,Lian,Barangay 1,0401013016
60549,NCR,Caloocan City,Pobcaran (Po,483615,Golden Minds Academy of Caloocan City Inc.,"No. 674 Pag-asa Street, Brgy. San Jose, Calooc...",CALOOCAN CITY,1st District,BARANGAY 131,private,urban,non_sectarian,purely_es,530,caloocan barangay 131,"[{'barangay': 'Barangay 131', 'municipality_or...",City of Caloocan,None,Barangay 131,1380100131
60550,NCR,Caloocan City,Pobcaran (Po,483644,Lightseekers Kiddie Center Inc.,"36 Massbielle St., 9th Avenue, Caloocan City",CALOOCAN CITY,2nd District,BARANGAY 59,private,urban,non_sectarian,purely_es,530,caloocan barangay 59,"[{'barangay': 'Barangay 5', 'municipality_or_c...",Batangas,Lian,Barangay 5,0401013020
60551,NCR,Caloocan City,Pobcaran (Po,483653,"Our Lady of Grace School of Caloocan, Inc.","357 J. Teodoro St., Corner 10th Ave., Gracepar...",CALOOCAN CITY,2nd District,BARANGAY 62,private,urban,non_sectarian,all_offering,530,caloocan barangay 62,"[{'barangay': 'Barangay 2', 'municipality_or_c...",Batangas,Lian,Barangay 2,0401013017


In [48]:
from barangay import search

In [49]:
r02_t001_searcher = partial(search, match_hooks=["province","barangay"], threshold=60.0, n=1, sanitizer=r02_t001_sanitizer)

In [50]:
r02_t001["score_r2"] = r02_t001["search_term"].apply(r02_t001_searcher)

/tmp/ipykernel_2284/4131783422.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["score_r2"] = r02_t001["search_term"].apply(r02_t001_searcher)


In [47]:
r02_t001[slim_view]

,barangay,municipality,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_id
45492,BARANGAY 161,CALOOCAN CITY,City of Caloocan,Barangay 161,None,1380100161
45493,BARANGAY 160,CALOOCAN CITY,City of Caloocan,Barangay 160,None,1380100160
45494,BARANGAY 147,CALOOCAN CITY,Pasay City,Barangay 147,None,1381100147
45495,BARANGAY 157,CALOOCAN CITY,Pasay City,Barangay 157,None,1381100157
45496,BARANGAY 83,CALOOCAN CITY,Batangas,Barangay 3,Lian,0401013018
...,...,...,...,...,...,...
60548,BARANGAY 18,CALOOCAN CITY,Batangas,Barangay 1,Lian,0401013016
60549,BARANGAY 131,CALOOCAN CITY,City of Caloocan,Barangay 131,None,1380100131
60550,BARANGAY 59,CALOOCAN CITY,Batangas,Barangay 5,Lian,0401013020
60551,BARANGAY 62,CALOOCAN CITY,Batangas,Barangay 2,Lian,0401013017


Get fucked again by ICC/HUC 🤬🤬🤬🤬
EDIT: or maybe not?? 🤔🤔

In [52]:
r02_t001["psgc_province_or_huc"] = r02_t001["score_r2"].apply(
    lambda x: x[0]["province_or_huc"] if len(x) > 0 else ""
)
r02_t001["psgc_municipality_or_city"] = r02_t001["score_r2"].apply(
    lambda x: x[0]["municipality_or_city"] if len(x) > 0 else ""
)
r02_t001["psgc_barangay"] = r02_t001["score_r2"].apply(
    lambda x: x[0]["barangay"] if len(x) > 0 else ""
)
r02_t001["psgc_id"] = r02_t001["score_r2"].apply(lambda x: x[0]["psgc_id"] if len(x) > 0 else "")

/tmp/ipykernel_2284/3462759361.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["psgc_province_or_huc"] = r02_t001["score_r2"].apply(
/tmp/ipykernel_2284/3462759361.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  r02_t001["psgc_municipality_or_city"] = r02_t001["score_r2"].apply(
/tmp/ipykernel_2284/3462759361.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docume

In [54]:
r02_t001[slim_view]

,barangay,municipality,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_id
45492,BARANGAY 161,CALOOCAN CITY,City of Caloocan,Barangay 161,None,1380100161
45493,BARANGAY 160,CALOOCAN CITY,City of Caloocan,Barangay 160,None,1380100160
45494,BARANGAY 147,CALOOCAN CITY,City of Caloocan,Barangay 147,None,1380100147
45495,BARANGAY 157,CALOOCAN CITY,City of Caloocan,Barangay 157,None,1380100157
45496,BARANGAY 83,CALOOCAN CITY,City of Caloocan,Barangay 83,None,1380100083
...,...,...,...,...,...,...
60548,BARANGAY 18,CALOOCAN CITY,City of Caloocan,Barangay 18,None,1380100018
60549,BARANGAY 131,CALOOCAN CITY,City of Caloocan,Barangay 131,None,1380100131
60550,BARANGAY 59,CALOOCAN CITY,City of Caloocan,Barangay 59,None,1380100059
60551,BARANGAY 62,CALOOCAN CITY,City of Caloocan,Barangay 62,None,1380100062


Now it works 😏
In summary, ed dataset run replace("Kalookan" with "Caloocan") and for Cities, we will use `pmb` match_hooks.

For round 3 setup: We'll now split the ed dataset into two sets, C => City and NC => non-city

# Round 3

In [12]:
import pandas as pd
from barangay import search, sanitize_input, FuzzBase
from functools import partial
from typing import Callable

# importing basic ed dataset
bed_df = pd.read_parquet("../data/education/basic_education_institutions.parquet")

In [13]:
bed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60924 entries, 0 to 60923
Data columns (total 14 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   region                                     60924 non-null  object
 1   division                                   60924 non-null  object
 2   district                                   60924 non-null  object
 3   beis_school_id                             60924 non-null  object
 4   school_name                                60924 non-null  object
 5   street_address                             60924 non-null  object
 6   municipality                               60924 non-null  object
 7   legislative_district                       60924 non-null  object
 8   barangay                                   60873 non-null  object
 9   sector                                     60924 non-null  object
 10  settlement_type                   

In [14]:
# Let's now separate the cities and non city
rows_with_city_of = bed_df["municipality"].str.lower().str.contains("city ")
rows_with_city = bed_df["municipality"].str.lower().str.contains(" city")

bed_df_c = bed_df[rows_with_city | rows_with_city_of]
bed_df_nc = bed_df[~ (rows_with_city | rows_with_city_of)]

In [15]:
# just double checking...
combined_rows_of_c_and_nc = len(bed_df_nc) + len(bed_df_c)
assert combined_rows_of_c_and_nc == 60924 # from the bed_df.info() above

City Terms

In [23]:
r3_sanitizer = partial(
    sanitize_input,
    exclude=[
        "(pob.)",
        "(pob)",
        "pob.",
        "(capital)",
        "city of ",
        " city",
        ".",
        "-",
        "(",
        ")",
        "&",
        ",",
    ],
)

fuzz_base = FuzzBase(sanitizer=r3_sanitizer)

def create_search_term_col(
    *, df: pd.DataFrame, sanitizer: Callable[..., str]
) -> pd.DataFrame:
    df["search_term"] = df["municipality"] + " " + df["barangay"]
    df["search_term"] = df["search_term"].apply(sanitizer)
    return df


def create_psgc_columns_from_match(
    *, df: pd.DataFrame, match_on_column: str
) -> pd.DataFrame:

    # breskeys: barangay results key
    breskeys = [
        "barangay",
        "province_or_huc",
        "municipality_or_city",
        "psgc_id",
        "f_0p0b_ratio",
        "f_00mb_ratio",
        "f_0pmb_ratio",
        "f_0p0b_ratio_score",
        "f_00mb_ratio_score",
        "f_0pmb_ratio_score",
        "0p0b",
        "00mb",
        "0pmb",
    ]
    for key in breskeys:
        df[f"psgc_{key}"] = df[match_on_column].apply(
            lambda x: x[0].get(key,"") if len(x) > 0 else ""
        )
    return df

slim_view = [
    "barangay",
    "municipality",
    "search_term",
    "psgc_province_or_huc",
    "psgc_barangay",
    "psgc_municipality_or_city",
    "psgc_psgc_id",
    # "f_0p0b_ratio",
    # "f_00mb_ratio",
    # "f_0pmb_ratio",
    "psgc_f_0p0b_ratio_score",
    "psgc_f_00mb_ratio_score",
    "psgc_f_0pmb_ratio_score",
    "psgc_0p0b",
    "psgc_00mb",
    "psgc_0pmb",

]

In [24]:
bed_df_c = create_search_term_col(df=bed_df_c, sanitizer=r3_sanitizer)

/tmp/ipykernel_32716/1704155661.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["search_term"] = df["municipality"] + " " + df["barangay"]
/tmp/ipykernel_32716/1704155661.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["search_term"] = df["search_term"].apply(sanitizer)


In [25]:
r3_c_search = partial(
    search,
    match_hooks=["province", "municipality", "barangay"],
    threshold=70,
    n=1,
    search_sanitizer=r3_sanitizer,
    fuzz_base=fuzz_base
)

In [26]:
# 45041 & 51585
sample001 = bed_df_c.loc[[45041,51585]]
sample001["match"] = sample001["search_term"].apply(r3_c_search)


In [27]:
sample001

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page,search_term,match
45041,CAR,Baguio City,District I,406228,Disciples for Christ Independent School Founda...,"Purok 4, Brgy Lucnab, Baguio City",BAGUIO CITY,Lone District,LUCNAB,private,urban,sectarian,all_offering,515,baguio lucnab,"[{'barangay': 'Lucnab', 'province_or_huc': 'Ci..."
51585,Region VI,Roxas City,Roxas City Dis,410347,ICAN Learning Center,"Martelino Subdivision, Arnaldo Boulevard",ROXAS CITY (Capital),1st District,BAYBAY,private,urban,non_sectarian,purely_es,264,roxas baybay,"[{'barangay': 'Baybay', 'province_or_huc': 'Ca..."


In [21]:
create_psgc_columns_from_match(df=sample001, match_on_column="match")[slim_view]

,barangay,municipality,search_term,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_psgc_id,psgc_f_0p0b_ratio_score,psgc_f_00mb_ratio_score,psgc_f_0pmb_ratio_score,psgc_0p0b,psgc_00mb,psgc_0pmb
45041,LUCNAB,BAGUIO CITY,baguio lucnab,City of Baguio,Lucnab,None,1430300064,100.000000,50.0,83.870968,baguio lucnab,none lucnab,baguio none lucnab
51585,BAYBAY,ROXAS CITY (Capital),roxas baybay,Capiz,Baybay,City of Roxas,0601914019,66.666667,100.0,80.000000,capiz baybay,roxas baybay,capiz roxas baybay


In [ ]:
# looks good

In [29]:
bed_df_c["match"] = bed_df_c["search_term"].apply(r3_c_search)

/tmp/ipykernel_32716/2768711018.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df_c["match"] = bed_df_c["search_term"].apply(r3_c_search)


In [40]:
create_psgc_columns_from_match(df=bed_df_c, match_on_column="match").sample(10)[slim_view]

/tmp/ipykernel_32716/1704155661.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f"psgc_{key}"] = df[match_on_column].apply(


,barangay,municipality,search_term,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_psgc_id,psgc_f_0p0b_ratio_score,psgc_f_00mb_ratio_score,psgc_f_0pmb_ratio_score,psgc_0p0b,psgc_00mb,psgc_0pmb
54506,BINCUNGAN,CITY OF TAGUM (Capital),tagum bincungan,Davao del Norte,Bincungan,City of Tagum,1102319003,55.0,100.0,65.217391,davao del norte bincungan,tagum bincungan,davao del norte tagum bincungan
57385,LOYOLA HEIGHTS,QUEZON CITY,quezon loyola heights,Quezon City,Loyola Heights,None,1381300055,100.0,85.0,89.361702,quezon loyola heights,none loyola heights,quezon none loyola heights
50590,SAN JOSE,IRIGA CITY,iriga san jose,Camarines Sur,San Jose,City of Iriga,0501716018,61.111111,100.0,66.666667,camarines sur san jose,iriga san jose,camarines sur iriga san jose
59168,LINAO NORTE,TUGUEGARAO CITY(Capital),tuguegarao linao norte,Cagayan,Linao Norte,Tuguegarao City,0201529046,53.658537,100.0,84.615385,cagayan linao norte,tuguegarao linao norte,cagayan tuguegarao linao norte
57465,DOÑA IMELDA,QUEZON CITY,quezon doña imelda,Quezon City,Doña Imelda,None,1381300031,100.0,82.352941,87.804878,quezon doña imelda,none doña imelda,quezon none doña imelda
33145,LIMPAPA,ZAMBOANGA CITY,zamboanga limpapa,City of Zamboanga,Limpapa,None,0931700041,100.0,68.965517,87.179487,zamboanga limpapa,none limpapa,zamboanga none limpapa
57092,COMMONWEALTH,QUEZON CITY,quezon commonwealth,Quezon City,Commonwealth,None,1381300022,100.0,83.333333,88.372093,quezon commonwealth,none commonwealth,quezon none commonwealth
55115,ROSARY HEIGHTS VII,COTABATO CITY,cotabato rosary heights vii,Maguindanao del Norte,Rosary Heights VII,City of Cotabato,1908703026,56.716418,100.0,71.052632,maguindanao del norte rosary heights vii,cotabato rosary heights vii,maguindanao del norte cotabato rosary heights vii
51617,BANICA,ROXAS CITY (Capital),roxas banica,Capiz,Banica,City of Roxas,0601914004,66.666667,100.0,80.0,capiz banica,roxas banica,capiz roxas banica
59537,POBLACION NO. 3 (BARANGAY 3),DUMAGUETE CITY (Capital),dumaguete poblacion no 3 barangay 3,Negros Oriental,Poblacion No. 3,City of Dumaguete,1804610021,52.307692,81.355932,64.0,negros oriental poblacion no 3,dumaguete poblacion no 3,negros oriental dumaguete poblacion no 3


In [41]:
for x in list(bed_df_c.columns):
    print(x)

region
division
district
beis_school_id
school_name
street_address
municipality
legislative_district
barangay
sector
settlement_type
school_subclassification
modified_cultural_offering_classification
masterlist_page
search_term
match
psgc_barangay
psgc_province_or_huc
psgc_municipality_or_city
psgc_psgc_id
psgc_f_0p0b_ratio
psgc_f_00mb_ratio
psgc_f_0pmb_ratio
psgc_f_0p0b_ratio_score
psgc_f_00mb_ratio_score
psgc_f_0pmb_ratio_score
psgc_0p0b
psgc_00mb
psgc_0pmb


In [49]:
bed_df_c_to_save = bed_df_c[
    [
        "region",
        "division",
        "district",
        "beis_school_id",
        "school_name",
        "street_address",
        "municipality",
        "legislative_district",
        "barangay",
        "sector",
        "settlement_type",
        "school_subclassification",
        "modified_cultural_offering_classification",
        "masterlist_page",
        "search_term",
        "match",
        "psgc_barangay",
        "psgc_province_or_huc",
        "psgc_municipality_or_city",
        "psgc_psgc_id",
        "psgc_f_0p0b_ratio_score",
        "psgc_f_00mb_ratio_score",
        "psgc_f_0pmb_ratio_score",
        "psgc_0p0b",
        "psgc_00mb",
        "psgc_0pmb",
    ]
].copy()
bed_df_c_to_save.astype(str).to_parquet("../data/education/basic_ed_matches_round03.parquet")

## For Non city

In [ ]:
import pandas as pd
from barangay import search, sanitize_input, FuzzBase
from functools import partial
from typing import Callable

# importing basic ed dataset
bed_df = pd.read_parquet("../data/education/basic_education_institutions.parquet")

# Let's now separate the cities and non city
rows_with_city_of = bed_df["municipality"].str.lower().str.contains("city ")
rows_with_city = bed_df["municipality"].str.lower().str.contains(" city")

bed_df_c = bed_df[rows_with_city | rows_with_city_of]
bed_df_nc = bed_df[~ (rows_with_city | rows_with_city_of)]

In [ ]:
# city checkpoint
bed_df_c = pd.read_parquet("../data/education/basic_ed_matches_round03.parquet")

In [51]:
r3_sanitizer = partial(
    sanitize_input,
    exclude=[
        "(pob.)",
        "(pob)",
        "pob.",
        "(capital)",
        "city of ",
        " city",
        ".",
        "-",
        "(",
        ")",
        "&",
        ",",
    ],
)

fuzz_base = FuzzBase(sanitizer=r3_sanitizer)

def create_search_term_col(
    *, df: pd.DataFrame, sanitizer: Callable[..., str]
) -> pd.DataFrame:
    df["search_term"] = df["municipality"] + " " + df["barangay"]
    df["search_term"] = df["search_term"].apply(sanitizer)
    return df


def create_psgc_columns_from_match(
    *, df: pd.DataFrame, match_on_column: str
) -> pd.DataFrame:

    # breskeys: barangay results key
    breskeys = [
        "barangay",
        "province_or_huc",
        "municipality_or_city",
        "psgc_id",
        "f_0p0b_ratio",
        "f_00mb_ratio",
        "f_0pmb_ratio",
        "f_0p0b_ratio_score",
        "f_00mb_ratio_score",
        "f_0pmb_ratio_score",
        "0p0b",
        "00mb",
        "0pmb",
    ]
    for key in breskeys:
        df[f"psgc_{key}"] = df[match_on_column].apply(
            lambda x: x[0].get(key,"") if len(x) > 0 else ""
        )
    return df

slim_view = [
    "barangay",
    "municipality",
    "search_term",
    "psgc_province_or_huc",
    "psgc_barangay",
    "psgc_municipality_or_city",
    "psgc_psgc_id",
    # "f_0p0b_ratio",
    # "f_00mb_ratio",
    # "f_0pmb_ratio",
    "psgc_f_0p0b_ratio_score",
    "psgc_f_00mb_ratio_score",
    "psgc_f_0pmb_ratio_score",
    "psgc_0p0b",
    "psgc_00mb",
    "psgc_0pmb",

]

In [54]:
r3_nc_search = partial(
    search,
    match_hooks=["municipality", "barangay"],
    threshold=70,
    n=1,
    search_sanitizer=r3_sanitizer,
    fuzz_base=fuzz_base
)

In [56]:
bed_df_nc = create_search_term_col(df=bed_df_nc, sanitizer=r3_sanitizer)

/tmp/ipykernel_32716/1704155661.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["search_term"] = df["municipality"] + " " + df["barangay"]
/tmp/ipykernel_32716/1704155661.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["search_term"] = df["search_term"].apply(sanitizer)


In [57]:
bed_df_nc["match"] = bed_df_nc["search_term"].apply(r3_nc_search)

/tmp/ipykernel_32716/938688598.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df_nc["match"] = bed_df_nc["search_term"].apply(r3_nc_search)


In [58]:
bed_df_nc = create_psgc_columns_from_match(df=bed_df_nc, match_on_column="match")

/tmp/ipykernel_32716/1704155661.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f"psgc_{key}"] = df[match_on_column].apply(
/tmp/ipykernel_32716/1704155661.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f"psgc_{key}"] = df[match_on_column].apply(
/tmp/ipykernel_32716/1704155661.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.

In [62]:
bed_df_nc[slim_view].sample(10)

,barangay,municipality,search_term,psgc_province_or_huc,psgc_barangay,psgc_municipality_or_city,psgc_psgc_id,psgc_f_0p0b_ratio_score,psgc_f_00mb_ratio_score,psgc_f_0pmb_ratio_score,psgc_0p0b,psgc_00mb,psgc_0pmb
9896,CAYSASAY,TAAL,taal caysasay,Batangas,Caysasay,Taal,0401029009,,100.0,,batangas caysasay,taal caysasay,batangas taal caysasay
17950,MALUBI,AROROY,aroroy malubi,Masbate,Malubi,Aroroy,0504101020,,100.0,,masbate malubi,aroroy malubi,masbate aroroy malubi
17659,COTMO,SIPOCOT,sipocot cotmo,Camarines Sur,Cotmo,Sipocot,0501734017,,100.0,,camarines sur cotmo,sipocot cotmo,camarines sur sipocot cotmo
3953,LINOMOT,JONES,jones linomot,Isabela,Linomot,Jones,0203115022,,100.0,,isabela linomot,jones linomot,isabela jones linomot
50237,SANTA MARIA (POB.),PRESENTACION (PARUBCAN),presentacion parubcan santa maria,Camarines Sur,Sta. Maria,Presentacion,0501729018,,80.0,,camarines sur sta maria,presentacion sta maria,camarines sur presentacion sta maria
4930,GREGORIO PIMENTEL,DIFFUN,diffun gregorio pimentel,Quirino,Gregorio Pimentel,Diffun,0205703032,,100.0,,quirino gregorio pimentel,diffun gregorio pimentel,quirino diffun gregorio pimentel
18433,DISTRICT I (POB.),SAN JACINTO,san jacinto district i,Masbate,District I,San Jacinto,0504119014,,100.0,,masbate district i,san jacinto district i,masbate san jacinto district i
55730,DEL CARMEN (POB.),DEL CARMEN,del carmen del carmen,Surigao del Norte,Del Carmen,Del Carmen,1606708009,,100.0,,surigao del norte del carmen,del carmen del carmen,surigao del norte del carmen del carmen
317,PARPAROROC,VINTAR,vintar parparoroc,Ilocos Norte,Parparoroc,Vintar,0102823039,,100.0,,ilocos norte parparoroc,vintar parparoroc,ilocos norte vintar parparoroc
15511,DUMAGUEÃ‘A,NARRA,narra dumagueã‘a,Palawan,Dumagueña,Narra,1705315008,,90.322581,,palawan dumagueña,narra dumagueña,palawan narra dumagueña
